In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_data["text"],
    train_data["target"],
    test_size=0.2,
    random_state=42,
    stratify=train_data["target"]
)

In [ ]:
print(X_train.shape)
print(X_val.shape)

The dataset is split into training and validation subsets using stratified sampling.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range = (1,5),
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train) # shape (6090, 10000)
X_val_tfidf = vectorizer.transform(X_val)

Tweets were transformed into sparse TF-IDF vectors including ngrams in range (1-5).

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42)

model.fit(X_train_tfidf, Y_train)

In [ ]:
preds = model.predict(X_val_tfidf)

In [ ]:
accuracy = accuracy_score(y_val, preds)
precision = precision_score(y_val, preds)
recall = recall_score(y_val, preds)
f1 = f1_score(y_val, preds)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
print(classification_report(y_val,preds))

In [ ]:
cm = confusion_matrix(
    y_val,
    preds
)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("TF-IDF + Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
coef = model.coef_[0]
feature_names = vectorizer.get_feature_names_out()
idx = np.argsort(coef)

In [ ]:
top_disaster = pd.DataFrame({
    'token': feature_names[idx[-20:]],
    'coef': coef[idx[-20:]]
}).sort_values('coef', ascending=False)

top_non_disaster = pd.DataFrame({
    'token': feature_names[idx[:20]],
    'coef': coef[idx[:20]]
}).sort_values('coef')


In [ ]:
top = 15

plt.figure(figsize=(10,6))

plt.barh(
    top_disaster["token"][:top],
    top_disaster["coef"][:top]
)

plt.title(
    "Most Disaster-related Tokens"
)

plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(
    feature_names[idx[:top]],
    coef[idx[:top]]
)
plt.title("Most Non-Disaster-related Tokens")
plt.show()

• TF-IDF with Logistic Regression achieved an F1 score of 0.775.

• Disaster-related keywords such as "earthquake", "evacuated", and "wildfire" received the highest positive coefficients.

• The model relies heavily on lexical signals and struggles with figurative language.

• This limitation motivates the use of contextual language models such as DistilBERT.


### Observations

Logistic Regression achieved the highest validation F1 score among the evaluated classical machine learning models.

Although XGBoost and LinearSVC performed competitively, neither provided a meaningful improvement over Logistic Regression.

Given its strong performance, simplicity, and interpretability, Logistic Regression was selected as the primary baseline model for further analysis and comparison with transformer-based approaches.